# RetinaNet

학습, 평가, 후처리, 제출 CSV 생성을 한 노트북에서 관리합니다.


In [ ]:
# Drive 마운트

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Colab 환경 준비

import os
!nvidia-smi

REPO_DIR = "/content/pill_detection_project"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/wina0901/pill_detection_project.git {REPO_DIR}
else:
    print("repo already exists:", REPO_DIR)

%cd {REPO_DIR}


In [ ]:
# import 및 공통 모듈 로드

import os
import sys
import json
import math
import time
import copy
import random
import shutil
import contextlib
import io
from pathlib import Path
from collections import defaultdict
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torchvision
import torchvision.transforms as T
from torchvision.ops import nms
from torchvision.models.detection import retinanet_resnet50_fpn
try:
    from torchvision.models.detection import retinanet_resnet50_fpn_v2
except Exception:
    retinanet_resnet50_fpn_v2 = None

from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import sigmoid_focal_loss

sys.path.append(REPO_DIR)
from src.preprocessing import (
    collate_fn,
    letterbox_with_bbox,
    show_samples,
    validate_coco,
)
from src.evaluation import (
    evaluate_all,
    convert_torchvision_outputs,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("DEVICE:", DEVICE)


## 경로 설정

실험, 결과 저장, 데이터셋 경로를 한 곳에서 정의합니다.


In [ ]:
# 프로젝트 경로 설정

def pick_existing_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return Path(candidates[0])

PROJECT_ROOT = Path("/content/drive/MyDrive/pill_detection_project")
RUNS_ROOT = PROJECT_ROOT / "models" / "retinanet"
RESULTS_ROOT = PROJECT_ROOT / "results" / "retinanet"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/content/drive/MyDrive/data/초급_프로젝트/dataset")
CP_ROOT = DATA_ROOT / "new_aug"

NO_CP = {
    "name": "NO_CP",
    "train_img": pick_existing_path([
        DATA_ROOT / "letterbox_images_aug_v1" / "train",
        DATA_ROOT / "letterbox_images_arg_v1" / "train",
    ]),
    "val_img": pick_existing_path([
        DATA_ROOT / "letterbox_images_aug_v1" / "val",
        DATA_ROOT / "letterbox_images_arg_v1" / "val",
    ]),
    "train_json": DATA_ROOT / "train_letterbox_aug_v1.json",
    "val_json": DATA_ROOT / "val_letterbox_aug_v1.json",
    "test_img": DATA_ROOT / "test_images",
}

CP = {
    "name": "CP",
    "train_img": CP_ROOT / "letterbox_images" / "train",
    "val_img": CP_ROOT / "letterbox_images" / "val",
    "train_json": CP_ROOT / "train_letterbox.json",
    "val_json": CP_ROOT / "val_letterbox.json",
    "test_img": DATA_ROOT / "test_images",
}

DATASETS = {
    "NO_CP": NO_CP,
    "CP": CP,
}

COMMON_CFG = {
    "img_size": 800,
    "batch_size": 4,
    "num_workers": 2,
    "seed": 42,
    "grad_clip_norm": 10.0,
}

BEST_DATASET_KEY = "NO_CP"

BEST_MODEL_CFG = {
    "backbone": "retinanet_resnet50_fpn",
    "weights": "DEFAULT",
    "anchor_preset": "default",
    "focal_alpha": 0.25,
    "focal_gamma": 3.0,
    "internal_score_thresh": 0.001,
    "internal_nms_thresh": 0.60,
    "detections_per_img": 300,
    "topk_candidates": 1000,
}

BEST_OPTIM_CFG = {
    "optimizer": "AdamW",
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 2,
    "min_lr": 1e-6,
}

BEST_FINETUNE_CFG = {
    "lr_stage_a": 3e-4,
    "lr_stage_b": 1e-4,
    "lr_stage_c": 3e-5,
}

BEST_POSTPROCESS_CFG = {
    "score_threshold": 0.20,
    "top_k_per_image": 4,
    "nms_iou_threshold": 0.50,
    "nms_mode": "agnostic",
    "pr_iou_threshold": 0.50,
}

RAW_EVAL_CFG = {
    "score_threshold": 0.001,
    "top_k_per_image": 300,
    "nms_iou_threshold": 0.60,
    "nms_mode": "classwise",
    "pr_iou_threshold": 0.50,
    "temp_json_name": "retinanet_raw_eval_temp.json",
}

SUBMISSION_EVAL_CFG = {
    "score_threshold": BEST_POSTPROCESS_CFG["score_threshold"],
    "top_k_per_image": BEST_POSTPROCESS_CFG["top_k_per_image"],
    "nms_iou_threshold": BEST_POSTPROCESS_CFG["nms_iou_threshold"],
    "nms_mode": BEST_POSTPROCESS_CFG["nms_mode"],
    "pr_iou_threshold": BEST_POSTPROCESS_CFG["pr_iou_threshold"],
    "temp_json_name": "retinanet_submission_eval_temp.json",
}

SWEEP_CFG = {
    "score_thresholds": [0.15, 0.20, 0.25, 0.30],
    "nms_iou_thresholds": [0.45, 0.50, 0.55],
    "nms_modes": [BEST_POSTPROCESS_CFG["nms_mode"]],
    "top_k_per_image": BEST_POSTPROCESS_CFG["top_k_per_image"],
    "pr_iou_threshold": BEST_POSTPROCESS_CFG["pr_iou_threshold"],
}

RUN_FLAGS = {
    "RUN_DATA_QC": True,
    "RUN_SCREENING": False,
    "RUN_DATASET_AB": False,
    "RUN_RETINA_ABLATION": False,
    "RUN_FINETUNE": False,
    "RUN_REPRO": False,
    "RUN_POSTPROCESS_SWEEP": False,
    "RUN_SUBMISSION_EXPORT": False,
    "RUN_CSV_ENSEMBLE": False,
}

MASTER_RESULTS_CSV = RESULTS_ROOT / "master_results.csv"

print("RUNS_ROOT   :", RUNS_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("NO_CP train :", NO_CP["train_img"])
print("CP train    :", CP["train_img"])


## 데이터 QC

학습 전 데이터 파일과 라벨 상태를 먼저 점검합니다.


In [ ]:
# 데이터 QC 함수

def load_json(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def summarize_coco_dataset(dataset_spec):
    train_coco = load_json(dataset_spec["train_json"])
    val_coco = load_json(dataset_spec["val_json"])

    def _summary(coco, image_root):
        image_root = Path(image_root)
        anns_by_image = defaultdict(list)
        for ann in coco.get("annotations", []):
            anns_by_image[int(ann["image_id"])].append(ann)

        image_ids = [int(img["id"]) for img in coco.get("images", [])]
        empty_images = sum(1 for image_id in image_ids if len(anns_by_image[image_id]) == 0)

        missing_files = []
        box_ws, box_hs, box_areas, box_ratios = [], [], [], []
        for img in coco.get("images", []):
            fp = image_root / Path(img["file_name"]).name
            if not fp.exists():
                missing_files.append(str(fp))
        for ann in coco.get("annotations", []):
            x, y, w, h = ann["bbox"]
            if w > 0 and h > 0:
                box_ws.append(float(w))
                box_hs.append(float(h))
                box_areas.append(float(w * h))
                box_ratios.append(float(w / h))

        category_ids = sorted({int(c["id"]) for c in coco.get("categories", [])})
        if not category_ids:
            category_ids = sorted({int(a["category_id"]) for a in coco.get("annotations", [])})

        return {
            "num_images": len(coco.get("images", [])),
            "num_annotations": len(coco.get("annotations", [])),
            "num_categories": len(category_ids),
            "empty_images": empty_images,
            "missing_files": len(missing_files),
            "missing_file_examples": missing_files[:5],
            "box_w_mean": float(np.mean(box_ws)) if box_ws else np.nan,
            "box_h_mean": float(np.mean(box_hs)) if box_hs else np.nan,
            "box_area_median": float(np.median(box_areas)) if box_areas else np.nan,
            "box_ratio_median": float(np.median(box_ratios)) if box_ratios else np.nan,
            "box_ws": box_ws,
            "box_hs": box_hs,
            "box_areas": box_areas,
            "box_ratios": box_ratios,
        }

    train_sum = _summary(train_coco, dataset_spec["train_img"])
    val_sum = _summary(val_coco, dataset_spec["val_img"])
    summary_df = pd.DataFrame([
        {"split": "train", **{k: v for k, v in train_sum.items() if not isinstance(v, list)}},
        {"split": "val", **{k: v for k, v in val_sum.items() if not isinstance(v, list)}},
    ])
    return summary_df, train_sum, val_sum, train_coco, val_coco

def plot_bbox_distributions(dataset_name, split_name, stats_dict):
    fig = plt.figure(figsize=(12, 8))

    ax1 = fig.add_subplot(2, 2, 1)
    ax1.hist(stats_dict["box_ws"], bins=40)
    ax1.set_title(f"{dataset_name} {split_name} bbox width")

    ax2 = fig.add_subplot(2, 2, 2)
    ax2.hist(stats_dict["box_hs"], bins=40)
    ax2.set_title(f"{dataset_name} {split_name} bbox height")

    ax3 = fig.add_subplot(2, 2, 3)
    areas = np.array(stats_dict["box_areas"], dtype=float)
    areas = areas[areas > 0]
    if len(areas):
        ax3.hist(np.log10(areas), bins=40)
    ax3.set_title(f"{dataset_name} {split_name} log10(area)")

    ax4 = fig.add_subplot(2, 2, 4)
    ratios = np.array(stats_dict["box_ratios"], dtype=float)
    ratios = ratios[np.isfinite(ratios)]
    if len(ratios):
        ax4.hist(np.clip(ratios, 0, 5), bins=40)
    ax4.set_title(f"{dataset_name} {split_name} aspect ratio (clipped)")
    plt.tight_layout()
    plt.show()

def visualize_coco_samples(dataset_spec, split="train", n=4):
    image_root = Path(dataset_spec["train_img"] if split == "train" else dataset_spec["val_img"])
    json_path = Path(dataset_spec["train_json"] if split == "train" else dataset_spec["val_json"])
    show_samples(
        img_dir=str(image_root),
        json_path=str(json_path),
        n=n,
        title=f"{dataset_spec['name']} {split}",
        show_bbox=True,
    )


In [ ]:
# 데이터 QC 실행

if RUN_FLAGS["RUN_DATA_QC"]:
    for dataset_name, dataset_spec in DATASETS.items():
        print("=" * 100)
        print("[DATASET]", dataset_name)
        validate_coco(str(dataset_spec["train_json"]), target_size=COMMON_CFG["img_size"])
        validate_coco(str(dataset_spec["val_json"]), target_size=COMMON_CFG["img_size"])
        summary_df, train_stats, val_stats, train_coco, val_coco = summarize_coco_dataset(dataset_spec)
        display(summary_df)
        plot_bbox_distributions(dataset_name, "train", train_stats)
        plot_bbox_distributions(dataset_name, "val", val_stats)
        visualize_coco_samples(dataset_spec, split="train", n=3)
        visualize_coco_samples(dataset_spec, split="val", n=3)
else:
    print("RUN_DATA_QC=False")


## Letterbox 기반 DataLoader

letterbox 전처리 기준으로 학습과 추론용 데이터를 구성합니다.


In [ ]:
# 시드, letterbox, DataLoader

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def apply_shared_letterbox(image, bboxes=None, target_size=800):
    bboxes = bboxes or []
    image_np = np.asarray(image)
    image_bgr = image_np[:, :, ::-1].copy()
    letterboxed_bgr, transformed_bboxes = letterbox_with_bbox(
        image_bgr,
        bboxes,
        target_size=target_size,
    )
    letterboxed = Image.fromarray(letterboxed_bgr[:, :, ::-1])

    orig_w, orig_h = image.size
    scale = target_size / max(orig_w, orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)
    pad_left = (target_size - new_w) // 2
    pad_top = (target_size - new_h) // 2
    meta = {
        "orig_w": orig_w,
        "orig_h": orig_h,
        "scale": scale,
        "pad_left": pad_left,
        "pad_top": pad_top,
        "new_w": new_w,
        "new_h": new_h,
    }
    return letterboxed, transformed_bboxes, meta

def letterbox_pil(image, target_size=800, fill=(114, 114, 114)):
    letterboxed, _, meta = apply_shared_letterbox(
        image,
        bboxes=[],
        target_size=target_size,
    )
    return letterboxed, meta

def xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x + w, y + h]

def restore_boxes_to_original_xyxy(boxes, meta):
    if boxes.numel() == 0:
        return boxes
    boxes = boxes.clone().float()
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - meta["pad_left"]) / meta["scale"]
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] - meta["pad_top"]) / meta["scale"]
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, meta["orig_w"])
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, meta["orig_h"])
    return boxes

class LetterboxCocoDetection(Dataset):
    def __init__(
        self,
        json_path,
        image_root,
        img_size=800,
        orig2model=None,
        training=True,
        with_annotations=True,
        image_ids=None,
    ):
        self.json_path = Path(json_path)
        self.image_root = Path(image_root)
        self.img_size = int(img_size)
        self.training = bool(training)
        self.with_annotations = bool(with_annotations)

        self.coco = load_json(self.json_path)
        self.images = sorted(self.coco.get("images", []), key=lambda x: int(x["id"]))
        if image_ids is not None:
            image_id_set = set(int(x) for x in image_ids)
            self.images = [img for img in self.images if int(img["id"]) in image_id_set]

        self.anns_by_image = defaultdict(list)
        for ann in self.coco.get("annotations", []):
            self.anns_by_image[int(ann["image_id"])].append(ann)

        category_ids = sorted({int(c["id"]) for c in self.coco.get("categories", [])})
        if not category_ids:
            category_ids = sorted({int(a["category_id"]) for a in self.coco.get("annotations", [])})

        self.orig2model = orig2model or {cat_id: idx for idx, cat_id in enumerate(category_ids)}
        self.model2orig = {v: k for k, v in self.orig2model.items()}
        self.num_classes = len(self.orig2model)

        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        image_id = int(img_info["id"])
        file_name = Path(img_info["file_name"]).name
        img_path = self.image_root / file_name

        image = Image.open(img_path).convert("RGB")
        anns = self.anns_by_image.get(image_id, [])
        valid_anns = []
        valid_bboxes = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            valid_anns.append(ann)
            valid_bboxes.append(ann["bbox"])

        letterboxed, transformed_bboxes, meta = apply_shared_letterbox(
            image,
            bboxes=valid_bboxes,
            target_size=self.img_size,
        )
        image_tensor = self.to_tensor(letterboxed)

        if not self.with_annotations:
            target = {
                "image_id": torch.tensor(image_id, dtype=torch.int64),
                "file_name": file_name,
                "letterbox_meta": meta,
                "orig_size": (image.height, image.width),
            }
            return image_tensor, target

        boxes, labels, areas, iscrowd = [], [], [], []
        for ann, final_bbox in zip(valid_anns, transformed_bboxes):
            if final_bbox is None:
                continue

            x, y, w, h = final_bbox
            x1 = float(x)
            y1 = float(y)
            x2 = float(x + w)
            y2 = float(y + h)

            x1 = max(0.0, min(float(self.img_size), x1))
            y1 = max(0.0, min(float(self.img_size), y1))
            x2 = max(0.0, min(float(self.img_size), x2))
            y2 = max(0.0, min(float(self.img_size), y2))
            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(int(self.orig2model[int(ann["category_id"])]))
            areas.append(float((x2 - x1) * (y2 - y1)))
            iscrowd.append(int(ann.get("iscrowd", 0)))

        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "area": areas,
            "iscrowd": iscrowd,
            "image_id": torch.tensor(image_id, dtype=torch.int64),
            "file_name": file_name,
            "letterbox_meta": meta,
            "orig_size": (image.height, image.width),
        }
        return image_tensor, target

def build_loaders_from_spec(dataset_spec, batch_size=4, num_workers=2, img_size=800):
    train_coco = load_json(dataset_spec["train_json"])
    category_ids = sorted({int(c["id"]) for c in train_coco.get("categories", [])})
    if not category_ids:
        category_ids = sorted({int(a["category_id"]) for a in train_coco.get("annotations", [])})
    orig2model = {cat_id: idx for idx, cat_id in enumerate(category_ids)}

    train_ds = LetterboxCocoDetection(
        json_path=dataset_spec["train_json"],
        image_root=dataset_spec["train_img"],
        img_size=img_size,
        orig2model=orig2model,
        training=True,
        with_annotations=True,
    )
    val_ds = LetterboxCocoDetection(
        json_path=dataset_spec["val_json"],
        image_root=dataset_spec["val_img"],
        img_size=img_size,
        orig2model=orig2model,
        training=False,
        with_annotations=True,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=collate_fn,
    )

    model2orig = {v: k for k, v in orig2model.items()}
    return train_loader, val_loader, orig2model, model2orig, train_ds.num_classes

train_loader, val_loader, orig2model, model2orig, num_classes = build_loaders_from_spec(
    DATASETS["NO_CP"],
    batch_size=COMMON_CFG["batch_size"],
    num_workers=COMMON_CFG["num_workers"],
    img_size=COMMON_CFG["img_size"],
)
print("num_classes:", num_classes)
print("train batches:", len(train_loader), "| val batches:", len(val_loader))


## 모델 정의

RetinaNet 모델 구성과 핵심 하이퍼파라미터를 정의합니다.


In [ ]:
# RetinaNet 모델 정의

class CustomRetinaNetClassificationHead(RetinaNetClassificationHead):
    def __init__(self, in_channels, num_anchors, num_classes, alpha=0.25, gamma=2.0, norm_layer=None):
        super().__init__(in_channels, num_anchors, num_classes, norm_layer=norm_layer)
        self.focal_alpha = alpha
        self.focal_gamma = gamma

    def compute_loss(self, targets, head_outputs, matched_idxs):
        cls_logits = head_outputs["cls_logits"]
        losses = []

        for targets_per_image, cls_logits_per_image, matched_idxs_per_image in zip(targets, cls_logits, matched_idxs):
            foreground_idxs = matched_idxs_per_image >= 0
            num_foreground = foreground_idxs.sum().item()

            gt_classes_target = torch.zeros_like(cls_logits_per_image)
            if num_foreground > 0:
                gt_classes_target[
                    foreground_idxs,
                    targets_per_image["labels"][matched_idxs_per_image[foreground_idxs]]
                ] = 1.0

            valid_idxs = matched_idxs_per_image != self.BETWEEN_THRESHOLDS
            loss = sigmoid_focal_loss(
                cls_logits_per_image[valid_idxs],
                gt_classes_target[valid_idxs],
                alpha=self.focal_alpha,
                gamma=self.focal_gamma,
                reduction="sum",
            ) / max(1, num_foreground)
            losses.append(loss)

        return sum(losses) / max(1, len(targets))

ANCHOR_PRESETS = {
    "default": None,
    "small_pill": {
        "sizes": ((16,), (32,), (64,), (128,), (256,)),
        "aspect_ratios": ((0.5, 1.0, 2.0),) * 5,
    },
    "wide_ratios": {
        "sizes": ((32,), (64,), (128,), (256,), (512,)),
        "aspect_ratios": ((0.33, 0.5, 1.0, 2.0, 3.0),) * 5,
    },
}

def build_anchor_generator(anchor_preset="default"):
    preset = ANCHOR_PRESETS.get(anchor_preset, None)
    if preset is None:
        return None
    return AnchorGenerator(
        sizes=tuple(tuple(x) for x in preset["sizes"]),
        aspect_ratios=tuple(tuple(x) for x in preset["aspect_ratios"]),
    )

def build_retinanet_model(num_classes, model_cfg):
    backbone_name = model_cfg.get("backbone", "retinanet_resnet50_fpn")
    weights = model_cfg.get("weights", "DEFAULT")
    anchor_preset = model_cfg.get("anchor_preset", "default")

    anchor_generator = build_anchor_generator(anchor_preset)

    if backbone_name == "retinanet_resnet50_fpn":
        model = retinanet_resnet50_fpn(weights=weights, anchor_generator=anchor_generator)
    elif backbone_name == "retinanet_resnet50_fpn_v2":
        if retinanet_resnet50_fpn_v2 is None:
            raise RuntimeError("retinanet_resnet50_fpn_v2 is not available in this torchvision version.")
        model = retinanet_resnet50_fpn_v2(weights=weights, anchor_generator=anchor_generator)
    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")

    cls_head = model.head.classification_head
    first_conv = next(module for module in cls_head.conv.modules() if isinstance(module, nn.Conv2d))
    in_channels = first_conv.in_channels
    num_anchors = cls_head.num_anchors

    alpha = model_cfg.get("focal_alpha", 0.25)
    gamma = model_cfg.get("focal_gamma", 2.0)

    custom_head = CustomRetinaNetClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
        alpha=alpha,
        gamma=gamma,
    )
    prior_probability = 0.01
    torch.nn.init.constant_(
        custom_head.cls_logits.bias,
        -math.log((1 - prior_probability) / prior_probability),
    )
    model.head.classification_head = custom_head

    if hasattr(model, "score_thresh"):
        model.score_thresh = float(model_cfg.get("internal_score_thresh", 0.001))
    if hasattr(model, "nms_thresh"):
        model.nms_thresh = float(model_cfg.get("internal_nms_thresh", 0.60))
    if hasattr(model, "detections_per_img"):
        model.detections_per_img = int(model_cfg.get("detections_per_img", 300))
    if hasattr(model, "topk_candidates"):
        model.topk_candidates = int(model_cfg.get("topk_candidates", 1000))

    return model

def apply_freeze_policy(model, freeze_cfg=None):
    freeze_cfg = freeze_cfg or {"scope": "none"}
    scope = freeze_cfg.get("scope", "none")

    for p in model.parameters():
        p.requires_grad = True

    if scope == "none":
        return

    if scope == "backbone":
        for p in model.backbone.parameters():
            p.requires_grad = False
    elif scope == "all_but_head":
        for p in model.parameters():
            p.requires_grad = False
        for p in model.head.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"Unsupported freeze scope: {scope}")

def build_optimizer_and_scheduler(model, optim_cfg, total_epochs):
    optimizer_name = optim_cfg.get("optimizer", "AdamW")
    lr = float(optim_cfg.get("lr", 1e-4))
    weight_decay = float(optim_cfg.get("weight_decay", 1e-4))
    warmup_epochs = int(optim_cfg.get("warmup_epochs", 2))
    min_lr = float(optim_cfg.get("min_lr", 1e-6))

    params = [p for p in model.parameters() if p.requires_grad]

    if optimizer_name == "AdamW":
        optimizer = optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "SGD":
        optimizer = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    warmup_epochs = min(warmup_epochs, max(1, total_epochs - 1))
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            optim.lr_scheduler.LinearLR(
                optimizer,
                start_factor=0.1,
                end_factor=1.0,
                total_iters=warmup_epochs,
            ),
            optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=max(1, total_epochs - warmup_epochs),
                eta_min=min_lr,
            ),
        ],
        milestones=[warmup_epochs],
    )
    return optimizer, scheduler


## 후처리 평가

raw 예측과 제출 스타일 예측을 같은 기준으로 비교합니다.


In [ ]:
# 후처리 함수

def apply_external_nms(boxes, scores, labels, iou_threshold=0.5, mode="agnostic"):
    if boxes.numel() == 0:
        return boxes, scores, labels

    if mode == "agnostic":
        keep = nms(boxes.float(), scores.float(), float(iou_threshold))
        return boxes[keep], scores[keep], labels[keep]

    if mode == "classwise":
        keep_indices = []
        unique_labels = labels.unique()
        for cls_id in unique_labels.tolist():
            idx = torch.where(labels == cls_id)[0]
            kept = nms(boxes[idx].float(), scores[idx].float(), float(iou_threshold))
            keep_indices.extend(idx[kept].tolist())
        keep_indices = sorted(keep_indices, key=lambda i: float(scores[i]), reverse=True)
        keep_indices = torch.tensor(keep_indices, dtype=torch.long)
        return boxes[keep_indices], scores[keep_indices], labels[keep_indices]

    raise ValueError(f"Unsupported NMS mode: {mode}")

def postprocess_prediction_tensors(
    boxes,
    scores,
    labels,
    score_threshold=BEST_POSTPROCESS_CFG["score_threshold"],
    top_k_per_image=BEST_POSTPROCESS_CFG["top_k_per_image"],
    nms_iou_threshold=BEST_POSTPROCESS_CFG["nms_iou_threshold"],
    nms_mode=BEST_POSTPROCESS_CFG["nms_mode"],
):
    if boxes.numel() == 0:
        return boxes, scores, labels

    keep = scores >= float(score_threshold)
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    if boxes.numel() == 0:
        return boxes, scores, labels

    boxes, scores, labels = apply_external_nms(
        boxes=boxes,
        scores=scores,
        labels=labels,
        iou_threshold=nms_iou_threshold,
        mode=nms_mode,
    )

    if top_k_per_image is not None and scores.numel() > int(top_k_per_image):
        scores, topk_idx = scores.topk(int(top_k_per_image))
        boxes = boxes[topk_idx]
        labels = labels[topk_idx]

    return boxes, scores, labels

def filter_prediction_dicts(
    predictions,
    score_threshold,
    top_k_per_image,
    nms_iou_threshold,
    nms_mode,
):
    by_image = defaultdict(list)
    for pred in predictions:
        by_image[int(pred["image_id"])].append(pred)

    filtered = []
    for image_id, preds in by_image.items():
        boxes = torch.tensor([pred["bbox_xyxy"] for pred in preds], dtype=torch.float32)
        scores = torch.tensor([pred["score"] for pred in preds], dtype=torch.float32)
        labels = torch.tensor([pred["category_id"] for pred in preds], dtype=torch.int64)

        boxes, scores, labels = postprocess_prediction_tensors(
            boxes=boxes,
            scores=scores,
            labels=labels,
            score_threshold=score_threshold,
            top_k_per_image=top_k_per_image,
            nms_iou_threshold=nms_iou_threshold,
            nms_mode=nms_mode,
        )

        for box, score, label in zip(boxes, scores, labels):
            filtered.append({
                "image_id": int(image_id),
                "category_id": int(label.item()),
                "bbox_xyxy": [float(x) for x in box.tolist()],
                "score": float(score.item()),
            })
    return filtered

def collect_model_outputs(model, loader, device=DEVICE):
    model.eval()
    all_outputs, all_image_ids = [], []
    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            outputs = model(images)
            batch_image_ids = [int(t["image_id"].item()) for t in targets]
            all_outputs.extend(outputs)
            all_image_ids.extend(batch_image_ids)
    return all_outputs, all_image_ids

def evaluate_prediction_list(gt_json_path, predictions, model2orig, conf_threshold, pr_iou_threshold=0.5, temp_json_path="temp_eval.json"):
    with contextlib.redirect_stdout(io.StringIO()):
        metrics = evaluate_all(
            gt_json_path=str(gt_json_path),
            predictions=predictions,
            conf_threshold=float(conf_threshold),
            pr_iou_threshold=float(pr_iou_threshold),
            temp_json_path=str(temp_json_path),
            model2orig=model2orig,
        )
    return metrics

def run_fixed_eval(model, loader, gt_json_path, model2orig, eval_cfg):
    all_outputs, all_image_ids = collect_model_outputs(model, loader, device=DEVICE)
    raw_predictions = convert_torchvision_outputs(all_outputs, all_image_ids)

    filtered_predictions = filter_prediction_dicts(
        predictions=raw_predictions,
        score_threshold=eval_cfg["score_threshold"],
        top_k_per_image=eval_cfg["top_k_per_image"],
        nms_iou_threshold=eval_cfg["nms_iou_threshold"],
        nms_mode=eval_cfg["nms_mode"],
    )

    metrics = evaluate_prediction_list(
        gt_json_path=gt_json_path,
        predictions=filtered_predictions,
        model2orig=model2orig,
        conf_threshold=eval_cfg["score_threshold"],
        pr_iou_threshold=eval_cfg["pr_iou_threshold"],
        temp_json_path=eval_cfg["temp_json_name"],
    )
    return metrics, filtered_predictions, raw_predictions

def run_postprocess_sweep(raw_predictions, gt_json_path, model2orig, sweep_cfg, temp_prefix="retinanet_sweep"):
    rows = []
    for thr, iou_thr, mode in product(
        sweep_cfg["score_thresholds"],
        sweep_cfg["nms_iou_thresholds"],
        sweep_cfg["nms_modes"],
    ):
        filtered_predictions = filter_prediction_dicts(
            predictions=raw_predictions,
            score_threshold=thr,
            top_k_per_image=sweep_cfg["top_k_per_image"],
            nms_iou_threshold=iou_thr,
            nms_mode=mode,
        )
        metrics = evaluate_prediction_list(
            gt_json_path=gt_json_path,
            predictions=filtered_predictions,
            model2orig=model2orig,
            conf_threshold=thr,
            pr_iou_threshold=sweep_cfg["pr_iou_threshold"],
            temp_json_path=f"{temp_prefix}_{mode}_{str(thr).replace('.', 'p')}_{str(iou_thr).replace('.', 'p')}.json",
        )
        rows.append({
            "score_threshold": thr,
            "top_k_per_image": sweep_cfg["top_k_per_image"],
            "nms_iou_threshold": iou_thr,
            "nms_mode": mode,
            "num_predictions": len(filtered_predictions),
            "mAP@75:95": metrics["mAP@75:95"],
            "mAP@50": metrics["mAP@50"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
        })

    df = pd.DataFrame(rows).sort_values(
        by=["mAP@75:95", "mAP@50", "precision", "recall"],
        ascending=False,
    ).reset_index(drop=True)
    return df

def flatten_metric_dict(prefix, metrics):
    out = {}
    for k, v in metrics.items():
        out[f"{prefix}_{k}"] = v
    return out


## 저장 유틸

체크포인트, history, summary 저장에 쓰는 헬퍼 함수입니다.


In [ ]:
# 체크포인트 저장 유틸

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_checkpoint_flexible(path, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        return ckpt["state_dict"], ckpt
    return ckpt, {"state_dict": ckpt}

def save_checkpoint(path, model, epoch, metric_name, metric_value, exp_cfg):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "state_dict": model.state_dict(),
        "epoch": int(epoch),
        "metric_name": str(metric_name),
        "metric_value": float(metric_value),
        "exp_name": exp_cfg["name"],
        "exp_cfg": exp_cfg,
    }, path)

def select_monitor_value(record, monitor_key):
    if monitor_key not in record:
        raise KeyError(f"monitor_key not found: {monitor_key}")
    return float(record[monitor_key])

def append_master_result(row, csv_path=MASTER_RESULTS_CSV):
    csv_path = Path(csv_path)
    if csv_path.exists():
        df_old = pd.read_csv(csv_path)
        df_new = pd.concat([df_old, pd.DataFrame([row])], ignore_index=True)
    else:
        df_new = pd.DataFrame([row])
    df_new.to_csv(csv_path, index=False)
    return df_new

def train_one_epoch(model, loader, optimizer, device=DEVICE, grad_clip_norm=None):
    model.train()
    loss_sum = 0.0
    n_batches = 0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: (v.to(device) if torch.is_tensor(v) else v) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(v for v in loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()

        loss_sum += float(loss.item())
        n_batches += 1

    return loss_sum / max(1, n_batches)

def evaluate_epoch_bundle(model, val_loader, gt_json_path, model2orig, raw_eval_cfg, submission_eval_cfg):
    raw_metrics, _, raw_predictions = run_fixed_eval(
        model=model,
        loader=val_loader,
        gt_json_path=gt_json_path,
        model2orig=model2orig,
        eval_cfg=raw_eval_cfg,
    )
    submission_metrics, _, _ = run_fixed_eval(
        model=model,
        loader=val_loader,
        gt_json_path=gt_json_path,
        model2orig=model2orig,
        eval_cfg=submission_eval_cfg,
    )
    return raw_metrics, submission_metrics, raw_predictions

def train_one_experiment(exp_cfg):
    exp_cfg = copy.deepcopy(exp_cfg)
    seed_everything(int(exp_cfg.get("seed", COMMON_CFG["seed"])))

    group = exp_cfg.get("group", "00_misc")
    run_dir = RUNS_ROOT / group / exp_cfg["name"]
    run_dir.mkdir(parents=True, exist_ok=True)

    dataset_key = exp_cfg["dataset_key"]
    dataset_spec = DATASETS[dataset_key]

    train_loader, val_loader, orig2model, model2orig, num_classes = build_loaders_from_spec(
        dataset_spec=dataset_spec,
        batch_size=int(exp_cfg.get("batch_size", COMMON_CFG["batch_size"])),
        num_workers=int(exp_cfg.get("num_workers", COMMON_CFG["num_workers"])),
        img_size=int(exp_cfg.get("img_size", COMMON_CFG["img_size"])),
    )

    model = build_retinanet_model(num_classes=num_classes, model_cfg=exp_cfg["model"])
    model.to(DEVICE)

    resume_ckpt = exp_cfg.get("resume_ckpt")
    if resume_ckpt:
        state_dict, meta = load_checkpoint_flexible(resume_ckpt, map_location=DEVICE)
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print("resume_ckpt loaded:", resume_ckpt)
        print("missing keys   :", len(missing))
        print("unexpected keys:", len(unexpected))

    apply_freeze_policy(model, exp_cfg.get("freeze"))

    optimizer, scheduler = build_optimizer_and_scheduler(
        model=model,
        optim_cfg=exp_cfg["optim"],
        total_epochs=int(exp_cfg["epochs"]),
    )

    history = []
    best_monitor = float("-inf")
    best_raw = float("-inf")
    best_submission = float("-inf")
    patience = int(exp_cfg.get("patience", 0))
    patience_counter = 0
    eval_every = int(exp_cfg.get("eval_every", 1))
    monitor_key = exp_cfg.get("monitor_key", "raw_mAP@75:95")
    start_time = time.time()

    save_json(exp_cfg, run_dir / "config.json")

    for epoch in range(1, int(exp_cfg["epochs"]) + 1):
        epoch_train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=DEVICE,
            grad_clip_norm=exp_cfg.get("grad_clip_norm", COMMON_CFG["grad_clip_norm"]),
        )

        row = {
            "epoch": epoch,
            "train_loss": epoch_train_loss,
            "lr": float(optimizer.param_groups[0]["lr"]),
        }

        if epoch % eval_every == 0:
            raw_metrics, submission_metrics, raw_predictions = evaluate_epoch_bundle(
                model=model,
                val_loader=val_loader,
                gt_json_path=dataset_spec["val_json"],
                model2orig=model2orig,
                raw_eval_cfg=exp_cfg.get("raw_eval_cfg", RAW_EVAL_CFG),
                submission_eval_cfg=exp_cfg.get("submission_eval_cfg", SUBMISSION_EVAL_CFG),
            )
            row.update(flatten_metric_dict("raw", raw_metrics))
            row.update(flatten_metric_dict("submission", submission_metrics))

            current_monitor = select_monitor_value(row, monitor_key)
            current_raw = float(raw_metrics["mAP@75:95"])
            current_submission = float(submission_metrics["mAP@75:95"])

            if current_monitor > best_monitor:
                best_monitor = current_monitor
                patience_counter = 0
                save_checkpoint(
                    run_dir / "best_monitor.pth",
                    model=model,
                    epoch=epoch,
                    metric_name=monitor_key,
                    metric_value=current_monitor,
                    exp_cfg=exp_cfg,
                )
            else:
                patience_counter += 1

            if current_raw > best_raw:
                best_raw = current_raw
                save_checkpoint(
                    run_dir / "best_raw.pth",
                    model=model,
                    epoch=epoch,
                    metric_name="raw_mAP@75:95",
                    metric_value=current_raw,
                    exp_cfg=exp_cfg,
                )

            if current_submission > best_submission:
                best_submission = current_submission
                save_checkpoint(
                    run_dir / "best_submission.pth",
                    model=model,
                    epoch=epoch,
                    metric_name="submission_mAP@75:95",
                    metric_value=current_submission,
                    exp_cfg=exp_cfg,
                )

            print(
                f"[{exp_cfg['name']}] "
                f"epoch {epoch:02d}/{exp_cfg['epochs']} | "
                f"train_loss={epoch_train_loss:.4f} | "
                f"raw_mAP@75:95={raw_metrics['mAP@75:95']:.4f} | "
                f"sub_mAP@75:95={submission_metrics['mAP@75:95']:.4f} | "
                f"monitor={current_monitor:.4f}"
            )

        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "metrics.csv", index=False)

        scheduler.step()

        if patience > 0 and patience_counter >= patience:
            print(f"Early stopping: patience={patience}")
            break

    elapsed_min = (time.time() - start_time) / 60.0
    save_json(history, run_dir / "history.json")
    save_checkpoint(
        run_dir / "last.pth",
        model=model,
        epoch=len(history),
        metric_name=monitor_key,
        metric_value=best_monitor if np.isfinite(best_monitor) else -1.0,
        exp_cfg=exp_cfg,
    )

    metrics_df = pd.DataFrame(history)
    last_row = metrics_df.iloc[-1].to_dict() if len(metrics_df) else {}

    summary_row = {
        "group": group,
        "run_name": exp_cfg["name"],
        "dataset_key": dataset_key,
        "backbone": exp_cfg["model"].get("backbone"),
        "anchor_preset": exp_cfg["model"].get("anchor_preset"),
        "focal_alpha": exp_cfg["model"].get("focal_alpha"),
        "focal_gamma": exp_cfg["model"].get("focal_gamma"),
        "internal_score_thresh": exp_cfg["model"].get("internal_score_thresh"),
        "internal_nms_thresh": exp_cfg["model"].get("internal_nms_thresh"),
        "detections_per_img": exp_cfg["model"].get("detections_per_img"),
        "optimizer": exp_cfg["optim"].get("optimizer"),
        "lr": exp_cfg["optim"].get("lr"),
        "epochs_requested": exp_cfg["epochs"],
        "epochs_ran": len(history),
        "monitor_key": monitor_key,
        "best_monitor": best_monitor,
        "best_raw_mAP@75:95": best_raw,
        "best_submission_mAP@75:95": best_submission,
        "elapsed_min": elapsed_min,
        "resume_ckpt": str(exp_cfg.get("resume_ckpt", "")),
        "best_monitor_ckpt": str(run_dir / "best_monitor.pth"),
        "best_raw_ckpt": str(run_dir / "best_raw.pth"),
        "best_submission_ckpt": str(run_dir / "best_submission.pth"),
        "metrics_csv": str(run_dir / "metrics.csv"),
    }
    summary_row.update(last_row)

    append_master_result(summary_row, csv_path=MASTER_RESULTS_CSV)
    save_json(summary_row, run_dir / "summary.json")
    return summary_row

def run_experiment_list(exp_list):
    rows = []
    for exp_cfg in exp_list:
        print("=" * 120)
        print("RUN:", exp_cfg["name"])
        rows.append(train_one_experiment(exp_cfg))
    df = pd.DataFrame(rows)
    return df

def load_master_results():
    if MASTER_RESULTS_CSV.exists():
        return pd.read_csv(MASTER_RESULTS_CSV)
    return pd.DataFrame()


## Screening

첫 번째 후보 설정을 빠르게 비교하는 실험입니다.


In [ ]:
# screening 실험 설정

SCREEN_GROUP = "01_screen"

SCREEN_EXPERIMENTS = [{
    "group": SCREEN_GROUP,
    "name": "screen_r50_nocp_lr0p0001_g3p0",
    "dataset_key": BEST_DATASET_KEY,
    "seed": COMMON_CFG["seed"],
    "img_size": COMMON_CFG["img_size"],
    "batch_size": COMMON_CFG["batch_size"],
    "num_workers": COMMON_CFG["num_workers"],
    "epochs": 8,
    "patience": 0,
    "eval_every": 1,
    "monitor_key": "raw_mAP@75:95",
    "model": copy.deepcopy(BEST_MODEL_CFG),
    "optim": copy.deepcopy(BEST_OPTIM_CFG),
    "freeze": {"scope": "none"},
    "raw_eval_cfg": RAW_EVAL_CFG,
    "submission_eval_cfg": SUBMISSION_EVAL_CFG,
}]

pd.DataFrame([
    {
        "name": x["name"],
        "dataset_key": x["dataset_key"],
        "lr": x["optim"]["lr"],
        "gamma": x["model"]["focal_gamma"],
        "epochs": x["epochs"],
    } for x in SCREEN_EXPERIMENTS
]).head(20)


In [ ]:
# screening 실행

if RUN_FLAGS["RUN_SCREENING"]:
    df_screen = run_experiment_list(SCREEN_EXPERIMENTS)
    display(df_screen.sort_values(["best_raw_mAP@75:95", "best_submission_mAP@75:95"], ascending=False))
else:
    print("RUN_SCREENING=False")


## Dataset A/B

데이터셋 구성에 따른 성능 차이를 비교합니다.


In [ ]:
# dataset A/B 실험 설정

AB_BASE_CONFIGS = [
    {
        "tag": "candA",
        "backbone": BEST_MODEL_CFG["backbone"],
        "lr": BEST_OPTIM_CFG["lr"],
        "gamma": BEST_MODEL_CFG["focal_gamma"],
        "anchor_preset": BEST_MODEL_CFG["anchor_preset"],
    },
]

AB_GROUP = "02_dataset_ab"
AB_EXPERIMENTS = []

for base_cfg in AB_BASE_CONFIGS:
    for dataset_key in ["NO_CP", "CP"]:
        lr_tag = str(base_cfg["lr"]).replace(".", "p")
        gamma_tag = str(base_cfg["gamma"]).replace(".", "p")
        AB_EXPERIMENTS.append({
            "group": AB_GROUP,
            "name": f"{base_cfg['tag']}_{dataset_key.lower()}_{base_cfg['backbone'].replace('retinanet_', '')}_lr{lr_tag}_g{gamma_tag}",
            "dataset_key": dataset_key,
            "seed": COMMON_CFG["seed"],
            "img_size": COMMON_CFG["img_size"],
            "batch_size": COMMON_CFG["batch_size"],
            "num_workers": COMMON_CFG["num_workers"],
            "epochs": 12,
            "patience": 0,
            "eval_every": 1,
            "monitor_key": "raw_mAP@75:95",
            "model": {
                "backbone": base_cfg["backbone"],
                "weights": "DEFAULT",
                "anchor_preset": base_cfg["anchor_preset"],
                "focal_alpha": 0.25,
                "focal_gamma": base_cfg["gamma"],
                "internal_score_thresh": 0.001,
                "internal_nms_thresh": 0.60,
                "detections_per_img": 300,
                "topk_candidates": 1000,
            },
            "optim": {
                "optimizer": "AdamW",
                "lr": base_cfg["lr"],
                "weight_decay": 1e-4,
                "warmup_epochs": 2,
                "min_lr": 1e-6,
            },
            "freeze": {"scope": "none"},
            "raw_eval_cfg": RAW_EVAL_CFG,
            "submission_eval_cfg": SUBMISSION_EVAL_CFG,
        })

pd.DataFrame([
    {
        "name": x["name"],
        "dataset_key": x["dataset_key"],
        "lr": x["optim"]["lr"],
        "gamma": x["model"]["focal_gamma"],
    } for x in AB_EXPERIMENTS
])


In [ ]:
# dataset A/B 실행

if RUN_FLAGS["RUN_DATASET_AB"]:
    df_ab = run_experiment_list(AB_EXPERIMENTS)
    display(df_ab.sort_values(["best_raw_mAP@75:95", "best_submission_mAP@75:95"], ascending=False))
else:
    print("RUN_DATASET_AB=False")


## RetinaNet 설정 비교

anchor와 내부 설정 변화에 따른 성능을 비교합니다.


In [ ]:
# RetinaNet 설정 실험

RETINA_ABLATION_GROUP = "03_retina_ablation"

RETINA_ABLATION_BASE = {
    "dataset_key": BEST_DATASET_KEY,
    "backbone": BEST_MODEL_CFG["backbone"],
    "lr": BEST_OPTIM_CFG["lr"],
    "gamma": BEST_MODEL_CFG["focal_gamma"],
    "anchor_preset": BEST_MODEL_CFG["anchor_preset"],
    "internal_score_thresh": BEST_MODEL_CFG["internal_score_thresh"],
    "detections_per_img": BEST_MODEL_CFG["detections_per_img"],
    "topk_candidates": BEST_MODEL_CFG["topk_candidates"],
}

RETINA_ABLATION_EXPERIMENTS = [{
    "group": RETINA_ABLATION_GROUP,
    "name": f"abl_{RETINA_ABLATION_BASE['dataset_key'].lower()}_{RETINA_ABLATION_BASE['anchor_preset']}_score{str(RETINA_ABLATION_BASE['internal_score_thresh']).replace('.', 'p')}_det{RETINA_ABLATION_BASE['detections_per_img']}",
    "dataset_key": RETINA_ABLATION_BASE["dataset_key"],
    "seed": COMMON_CFG["seed"],
    "img_size": COMMON_CFG["img_size"],
    "batch_size": COMMON_CFG["batch_size"],
    "num_workers": COMMON_CFG["num_workers"],
    "epochs": 10,
    "patience": 0,
    "eval_every": 1,
    "monitor_key": "raw_mAP@75:95",
    "model": {
        "backbone": RETINA_ABLATION_BASE["backbone"],
        "weights": "DEFAULT",
        "anchor_preset": RETINA_ABLATION_BASE["anchor_preset"],
        "focal_alpha": BEST_MODEL_CFG["focal_alpha"],
        "focal_gamma": RETINA_ABLATION_BASE["gamma"],
        "internal_score_thresh": RETINA_ABLATION_BASE["internal_score_thresh"],
        "internal_nms_thresh": BEST_MODEL_CFG["internal_nms_thresh"],
        "detections_per_img": RETINA_ABLATION_BASE["detections_per_img"],
        "topk_candidates": RETINA_ABLATION_BASE["topk_candidates"],
    },
    "optim": copy.deepcopy(BEST_OPTIM_CFG),
    "freeze": {"scope": "none"},
    "raw_eval_cfg": RAW_EVAL_CFG,
    "submission_eval_cfg": SUBMISSION_EVAL_CFG,
}]

pd.DataFrame([
    {
        "name": x["name"],
        "anchor_preset": x["model"]["anchor_preset"],
        "internal_score_thresh": x["model"]["internal_score_thresh"],
        "detections_per_img": x["model"]["detections_per_img"],
    } for x in RETINA_ABLATION_EXPERIMENTS
]).head(20)


In [ ]:
# RetinaNet 설정 비교 실행

if RUN_FLAGS["RUN_RETINA_ABLATION"]:
    df_retina_ablation = run_experiment_list(RETINA_ABLATION_EXPERIMENTS)
    display(df_retina_ablation.sort_values(["best_raw_mAP@75:95", "best_submission_mAP@75:95"], ascending=False))
else:
    print("RUN_RETINA_ABLATION=False")


## Staged Fine-tuning

head -> backbone 고정 -> 전체 unfreeze 순서로 미세조정합니다.


In [ ]:
# staged fine-tuning 설정

FINETUNE_GROUP = "04_finetune"

FINETUNE_BASE_CKPTS = [
    {
        "tag": "best_single",
        "dataset_key": BEST_DATASET_KEY,
        "resume_ckpt": str(RUNS_ROOT / "02_dataset_ab" / "candA_no_cp_resnet50_fpn_lr0p0001_g3p0" / "best_monitor.pth"),
        "backbone": BEST_MODEL_CFG["backbone"],
        "lr_stage_a": BEST_FINETUNE_CFG["lr_stage_a"],
        "lr_stage_b": BEST_FINETUNE_CFG["lr_stage_b"],
        "lr_stage_c": BEST_FINETUNE_CFG["lr_stage_c"],
        "gamma": BEST_MODEL_CFG["focal_gamma"],
        "anchor_preset": BEST_MODEL_CFG["anchor_preset"],
    },
]

def build_staged_finetune_experiments(base_row):
    tag = base_row["tag"]
    dataset_key = base_row["dataset_key"]
    backbone = base_row["backbone"]
    gamma = base_row["gamma"]
    anchor_preset = base_row["anchor_preset"]
    resume_ckpt = base_row["resume_ckpt"]

    return [
        {
            "group": FINETUNE_GROUP,
            "name": f"{tag}_stageA_head_only",
            "dataset_key": dataset_key,
            "seed": COMMON_CFG["seed"],
            "img_size": COMMON_CFG["img_size"],
            "batch_size": COMMON_CFG["batch_size"],
            "num_workers": COMMON_CFG["num_workers"],
            "epochs": 3,
            "patience": 0,
            "eval_every": 1,
            "monitor_key": "submission_mAP@75:95",
            "resume_ckpt": resume_ckpt,
            "model": {
                "backbone": backbone,
                "weights": "DEFAULT",
                "anchor_preset": anchor_preset,
                "focal_alpha": 0.25,
                "focal_gamma": gamma,
                "internal_score_thresh": 0.001,
                "internal_nms_thresh": 0.60,
                "detections_per_img": 300,
                "topk_candidates": 1000,
            },
            "optim": {
                "optimizer": "AdamW",
                "lr": base_row["lr_stage_a"],
                "weight_decay": 1e-4,
                "warmup_epochs": 1,
                "min_lr": 1e-6,
            },
            "freeze": {"scope": "all_but_head"},
            "raw_eval_cfg": RAW_EVAL_CFG,
            "submission_eval_cfg": SUBMISSION_EVAL_CFG,
        },
        {
            "group": FINETUNE_GROUP,
            "name": f"{tag}_stageB_backbone_frozen",
            "dataset_key": dataset_key,
            "seed": COMMON_CFG["seed"],
            "img_size": COMMON_CFG["img_size"],
            "batch_size": COMMON_CFG["batch_size"],
            "num_workers": COMMON_CFG["num_workers"],
            "epochs": 8,
            "patience": 3,
            "eval_every": 1,
            "monitor_key": "submission_mAP@75:95",
            "resume_ckpt": str(RUNS_ROOT / FINETUNE_GROUP / f"{tag}_stageA_head_only" / "best_monitor.pth"),
            "model": {
                "backbone": backbone,
                "weights": "DEFAULT",
                "anchor_preset": anchor_preset,
                "focal_alpha": 0.25,
                "focal_gamma": gamma,
                "internal_score_thresh": 0.001,
                "internal_nms_thresh": 0.60,
                "detections_per_img": 300,
                "topk_candidates": 1000,
            },
            "optim": {
                "optimizer": "AdamW",
                "lr": base_row["lr_stage_b"],
                "weight_decay": 1e-4,
                "warmup_epochs": 1,
                "min_lr": 1e-6,
            },
            "freeze": {"scope": "backbone"},
            "raw_eval_cfg": RAW_EVAL_CFG,
            "submission_eval_cfg": SUBMISSION_EVAL_CFG,
        },
        {
            "group": FINETUNE_GROUP,
            "name": f"{tag}_stageC_full_unfreeze",
            "dataset_key": dataset_key,
            "seed": COMMON_CFG["seed"],
            "img_size": COMMON_CFG["img_size"],
            "batch_size": COMMON_CFG["batch_size"],
            "num_workers": COMMON_CFG["num_workers"],
            "epochs": 6,
            "patience": 3,
            "eval_every": 1,
            "monitor_key": "submission_mAP@75:95",
            "resume_ckpt": str(RUNS_ROOT / FINETUNE_GROUP / f"{tag}_stageB_backbone_frozen" / "best_monitor.pth"),
            "model": {
                "backbone": backbone,
                "weights": "DEFAULT",
                "anchor_preset": anchor_preset,
                "focal_alpha": 0.25,
                "focal_gamma": gamma,
                "internal_score_thresh": 0.001,
                "internal_nms_thresh": 0.60,
                "detections_per_img": 300,
                "topk_candidates": 1000,
            },
            "optim": {
                "optimizer": "AdamW",
                "lr": base_row["lr_stage_c"],
                "weight_decay": 1e-4,
                "warmup_epochs": 1,
                "min_lr": 1e-6,
            },
            "freeze": {"scope": "none"},
            "raw_eval_cfg": RAW_EVAL_CFG,
            "submission_eval_cfg": SUBMISSION_EVAL_CFG,
        },
    ]

FINETUNE_EXPERIMENTS = []
for row in FINETUNE_BASE_CKPTS:
    FINETUNE_EXPERIMENTS.extend(build_staged_finetune_experiments(row))

pd.DataFrame([
    {
        "name": x["name"],
        "resume_ckpt": x.get("resume_ckpt", ""),
        "freeze_scope": x["freeze"]["scope"],
        "lr": x["optim"]["lr"],
        "epochs": x["epochs"],
    } for x in FINETUNE_EXPERIMENTS
])


In [ ]:
# staged fine-tuning 실행

if RUN_FLAGS["RUN_FINETUNE"]:
    df_ft = run_experiment_list(FINETUNE_EXPERIMENTS)
    display(df_ft.sort_values(["best_submission_mAP@75:95", "best_raw_mAP@75:95"], ascending=False))
else:
    print("RUN_FINETUNE=False")


## Postprocess Sweep

제출 후처리에 쓰는 threshold와 NMS 조합을 탐색합니다.


In [ ]:
# postprocess sweep 실행 함수

def build_model_and_load_ckpt(ckpt_path, dataset_key, model_cfg):
    _, val_loader, orig2model, model2orig, num_classes = build_loaders_from_spec(
        dataset_spec=DATASETS[dataset_key],
        batch_size=COMMON_CFG["batch_size"],
        num_workers=COMMON_CFG["num_workers"],
        img_size=COMMON_CFG["img_size"],
    )
    model = build_retinanet_model(num_classes=num_classes, model_cfg=model_cfg)
    state_dict, meta = load_checkpoint_flexible(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state_dict, strict=False)
    model.to(DEVICE)
    model.eval()
    return model, val_loader, model2orig

POSTPROCESS_TARGETS = [
    {
        "name": "best_single_target",
        "dataset_key": BEST_DATASET_KEY,
        "ckpt_path": str(RUNS_ROOT / "04_finetune" / "best_single_stageC_full_unfreeze" / "best_monitor.pth"),
        "model_cfg": copy.deepcopy(BEST_MODEL_CFG),
    },
]

if RUN_FLAGS["RUN_POSTPROCESS_SWEEP"]:
    for target in POSTPROCESS_TARGETS:
        model, val_loader_pp, model2orig_pp = build_model_and_load_ckpt(
            ckpt_path=target["ckpt_path"],
            dataset_key=target["dataset_key"],
            model_cfg=target["model_cfg"],
        )
        all_outputs, all_image_ids = collect_model_outputs(model, val_loader_pp, device=DEVICE)
        raw_predictions = convert_torchvision_outputs(all_outputs, all_image_ids)

        df_sweep = run_postprocess_sweep(
            raw_predictions=raw_predictions,
            gt_json_path=DATASETS[target["dataset_key"]]["val_json"],
            model2orig=model2orig_pp,
            sweep_cfg=SWEEP_CFG,
            temp_prefix=target["name"],
        )
        sweep_csv = RESULTS_ROOT / f"{target['name']}_postprocess_sweep.csv"
        df_sweep.to_csv(sweep_csv, index=False)
        print("saved:", sweep_csv)
        display(df_sweep.head(10))
else:
    print("RUN_POSTPROCESS_SWEEP=False")


## Submission Export

선택한 체크포인트로 제출용 CSV를 생성합니다.


In [ ]:
# 제출 CSV 생성

def infer_single_image(model, image_path, img_size=800):
    image = Image.open(image_path).convert("RGB")
    letterboxed, meta = letterbox_pil(image, target_size=img_size)
    tensor = T.ToTensor()(letterboxed).to(DEVICE)

    with torch.no_grad():
        output = model([tensor])[0]

    boxes = output["boxes"].detach().cpu()
    scores = output["scores"].detach().cpu()
    labels = output["labels"].detach().cpu()
    boxes = restore_boxes_to_original_xyxy(boxes, meta)
    return boxes, scores, labels

def make_submission_csv(
    ckpt_path,
    dataset_key,
    model_cfg,
    output_csv_path,
    score_threshold=BEST_POSTPROCESS_CFG["score_threshold"],
    top_k_per_image=BEST_POSTPROCESS_CFG["top_k_per_image"],
    nms_iou_threshold=BEST_POSTPROCESS_CFG["nms_iou_threshold"],
    nms_mode=BEST_POSTPROCESS_CFG["nms_mode"],
):
    dataset_spec = DATASETS[dataset_key]
    _, _, orig2model_sub, model2orig_sub, num_classes_sub = build_loaders_from_spec(
        dataset_spec=dataset_spec,
        batch_size=COMMON_CFG["batch_size"],
        num_workers=COMMON_CFG["num_workers"],
        img_size=COMMON_CFG["img_size"],
    )
    model = build_retinanet_model(num_classes=num_classes_sub, model_cfg=model_cfg)
    state_dict, meta = load_checkpoint_flexible(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state_dict, strict=False)
    model.to(DEVICE)
    model.eval()

    test_dir = Path(dataset_spec["test_img"])
    test_files = sorted([p for p in test_dir.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])

    rows = []
    for fp in test_files:
        image_id = fp.stem
        boxes, scores, labels = infer_single_image(model, fp, img_size=COMMON_CFG["img_size"])
        boxes, scores, labels = postprocess_prediction_tensors(
            boxes=boxes,
            scores=scores,
            labels=labels,
            score_threshold=score_threshold,
            top_k_per_image=top_k_per_image,
            nms_iou_threshold=nms_iou_threshold,
            nms_mode=nms_mode,
        )

        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.tolist()
            orig_cat = model2orig_sub[int(label.item())]
            rows.append({
                "image_id": image_id,
                "category_id": int(orig_cat + 1),
                "bbox_x": float(x1),
                "bbox_y": float(y1),
                "bbox_w": float(x2 - x1),
                "bbox_h": float(y2 - y1),
                "score": float(score.item()),
            })

    if rows:
        df_sub = pd.DataFrame(rows).sort_values(["image_id", "score"], ascending=[True, False]).reset_index(drop=True)
        df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))
    else:
        df_sub = pd.DataFrame(columns=[
            "annotation_id", "image_id", "category_id",
            "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
        ])

    output_csv_path = Path(output_csv_path)
    output_csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_sub.to_csv(output_csv_path, index=False)
    print("saved:", output_csv_path, "| rows:", len(df_sub))
    return df_sub

SUBMISSION_TARGETS = [
    {
        "name": "retinanet_best_single",
        "dataset_key": BEST_DATASET_KEY,
        "ckpt_path": str(RUNS_ROOT / "04_finetune" / "best_single_stageC_full_unfreeze" / "best_monitor.pth"),
        "model_cfg": copy.deepcopy(BEST_MODEL_CFG),
        "postprocess": copy.deepcopy(BEST_POSTPROCESS_CFG)
    }
]

if RUN_FLAGS["RUN_SUBMISSION_EXPORT"]:
    for target in SUBMISSION_TARGETS:
        output_csv = RESULTS_ROOT / f"{target['name']}_submission.csv"
        df_sub = make_submission_csv(
            ckpt_path=target["ckpt_path"],
            dataset_key=target["dataset_key"],
            model_cfg=target["model_cfg"],
            output_csv_path=output_csv,
            **target["postprocess"],
        )
        display(df_sub.head())
else:
    print("RUN_SUBMISSION_EXPORT=False")


## CSV Ensemble

여러 제출 CSV를 다시 합쳐 후처리하는 단계입니다.


In [ ]:
# CSV ensemble

def dataframe_to_xyxy(df):
    x1 = df["bbox_x"].to_numpy(dtype=float)
    y1 = df["bbox_y"].to_numpy(dtype=float)
    x2 = x1 + df["bbox_w"].to_numpy(dtype=float)
    y2 = y1 + df["bbox_h"].to_numpy(dtype=float)
    return np.stack([x1, y1, x2, y2], axis=1)

def ensemble_submission_csvs(
    csv_paths,
    output_csv_path,
    weights=None,
    score_threshold=0.0,
    top_k_per_image=4,
    nms_iou_threshold=0.55,
    nms_mode="classwise",
):
    csv_paths = [Path(p) for p in csv_paths]
    weights = weights or [1.0] * len(csv_paths)
    assert len(csv_paths) == len(weights)

    frames = []
    for csv_path, w in zip(csv_paths, weights):
        df = pd.read_csv(csv_path).copy()
        if "score" not in df.columns:
            raise ValueError(f"score column missing: {csv_path}")
        df["score"] = df["score"].astype(float) * float(w)
        df["source_csv"] = csv_path.name
        frames.append(df)

    if not frames:
        raise ValueError("No csv paths provided")

    merged = pd.concat(frames, ignore_index=True)
    output_rows = []

    for image_id, group in merged.groupby("image_id"):
        boxes = torch.tensor(dataframe_to_xyxy(group), dtype=torch.float32)
        scores = torch.tensor(group["score"].to_numpy(dtype=float), dtype=torch.float32)
        labels = torch.tensor(group["category_id"].to_numpy(dtype=int), dtype=torch.int64)

        boxes, scores, labels = postprocess_prediction_tensors(
            boxes=boxes,
            scores=scores,
            labels=labels,
            score_threshold=score_threshold,
            top_k_per_image=top_k_per_image,
            nms_iou_threshold=nms_iou_threshold,
            nms_mode=nms_mode,
        )

        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.tolist()
            output_rows.append({
                "image_id": image_id,
                "category_id": int(label.item()),
                "bbox_x": float(x1),
                "bbox_y": float(y1),
                "bbox_w": float(x2 - x1),
                "bbox_h": float(y2 - y1),
                "score": float(score.item()),
            })

    if output_rows:
        df_out = pd.DataFrame(output_rows).sort_values(["image_id", "score"], ascending=[True, False]).reset_index(drop=True)
        df_out.insert(0, "annotation_id", range(1, len(df_out) + 1))
    else:
        df_out = pd.DataFrame(columns=[
            "annotation_id", "image_id", "category_id",
            "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
        ])

    output_csv_path = Path(output_csv_path)
    output_csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(output_csv_path, index=False)
    print("saved:", output_csv_path, "| rows:", len(df_out))
    return df_out

ENSEMBLE_PLAN = {
    "csv_paths": [
        RESULTS_ROOT / "retinanet_best_single_submission.csv",
    ],
    "weights": [1.0],
    "output_csv_path": RESULTS_ROOT / "retinanet_csv_ensemble_submission.csv",
    "score_threshold": 0.0,
    "top_k_per_image": 4,
    "nms_iou_threshold": 0.55,
    "nms_mode": "classwise",
}

if RUN_FLAGS["RUN_CSV_ENSEMBLE"]:
    df_ens = ensemble_submission_csvs(**ENSEMBLE_PLAN)
    display(df_ens.head())
else:
    print("RUN_CSV_ENSEMBLE=False")


## Repro

seed에 따른 재현성을 확인합니다.


In [ ]:
# 재현 실험 설정

REPRO_GROUP = "05_repro"

REPRO_BASE_EXPERIMENT = {
    "name_prefix": "repro_best_single",
    "dataset_key": BEST_DATASET_KEY,
    "img_size": COMMON_CFG["img_size"],
    "batch_size": COMMON_CFG["batch_size"],
    "num_workers": COMMON_CFG["num_workers"],
    "epochs": 10,
    "patience": 3,
    "eval_every": 1,
    "monitor_key": "submission_mAP@75:95",
    "resume_ckpt": str(RUNS_ROOT / "04_finetune" / "best_single_stageC_full_unfreeze" / "best_monitor.pth"),
    "model": copy.deepcopy(BEST_MODEL_CFG),
    "optim": {
        "optimizer": BEST_OPTIM_CFG["optimizer"],
        "lr": BEST_FINETUNE_CFG["lr_stage_c"],
        "weight_decay": BEST_OPTIM_CFG["weight_decay"],
        "warmup_epochs": 1,
        "min_lr": BEST_OPTIM_CFG["min_lr"],
    },
    "freeze": {"scope": "none"},
    "raw_eval_cfg": RAW_EVAL_CFG,
    "submission_eval_cfg": SUBMISSION_EVAL_CFG,
}

REPRO_EXPERIMENTS = []
for seed in [42, 3407, 2026]:
    item = copy.deepcopy(REPRO_BASE_EXPERIMENT)
    item["group"] = REPRO_GROUP
    item["seed"] = seed
    item["name"] = f"{REPRO_BASE_EXPERIMENT['name_prefix']}_seed{seed}"
    REPRO_EXPERIMENTS.append(item)

pd.DataFrame([
    {"name": x["name"], "seed": x["seed"], "resume_ckpt": x["resume_ckpt"]} for x in REPRO_EXPERIMENTS
])


In [ ]:
# 재현 실험 실행

if RUN_FLAGS["RUN_REPRO"]:
    df_repro = run_experiment_list(REPRO_EXPERIMENTS)
    display(df_repro)

    df_repro_summary = df_repro[[
        "run_name",
        "best_raw_mAP@75:95",
        "best_submission_mAP@75:95",
        "elapsed_min",
    ]].copy()

    print("mean raw mAP@75:95        :", df_repro_summary["best_raw_mAP@75:95"].mean())
    print("std raw mAP@75:95         :", df_repro_summary["best_raw_mAP@75:95"].std())
    print("mean submission mAP@75:95 :", df_repro_summary["best_submission_mAP@75:95"].mean())
    print("std submission mAP@75:95  :", df_repro_summary["best_submission_mAP@75:95"].std())
else:
    print("RUN_REPRO=False")


## 결과 요약

실험별 주요 성능을 한 표로 정리합니다.


In [ ]:
# 결과 요약

master_df = load_master_results()
print("master rows:", len(master_df))
if len(master_df):
    cols = [c for c in [
        "group", "run_name", "dataset_key", "backbone", "anchor_preset",
        "focal_gamma", "lr", "best_raw_mAP@75:95", "best_submission_mAP@75:95",
        "elapsed_min", "best_monitor_ckpt"
    ] if c in master_df.columns]
    display(master_df[cols].sort_values(["best_submission_mAP@75:95", "best_raw_mAP@75:95"], ascending=False).head(50))
else:
    print("No master results yet.")
